<img src="https://d1yjjnpx0p53s8.cloudfront.net/styles/logo-thumbnail/s3/102012/logo_unab.png?itok=hZ5x30O2" width="240" height="240" align="right"/>

<center><h1>Statistics for Data Science</h1></center>
<left><h1>Unit 1: Exploratory Data Analysis</h1></left>

# Case Study: Morphological Differentiation Across Palmer Penguin Species

**Author:** Pedro Esteban Martínez Santamaría  
**Date:** June 2026  
**Course:** Statistics for Data Science — M.Sc. Data Science, UNAB

## General Description

This study performs an Exploratory Data Analysis (EDA) on the **Palmer Penguins** dataset to investigate morphological differences across three penguin species (Adélie, Chinstrap, Gentoo) collected from three islands in the Palmer Archipelago, Antarctica.

The goal is to identify patterns, trends, and statistical relationships between physical measurements (bill length, bill depth, flipper length, body mass) and categorical variables (species, island, sex) that could support species classification based on morphology alone.

---

## Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats

sns.set_theme(style="whitegrid", palette="colorblind")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["figure.dpi"] = 100

%matplotlib inline

## Load the Dataset

In [ ]:
URL = "https://raw.githubusercontent.com/PedroMartk9i/Penguins-Palmer-Own-Format/main/penguins_size%20(1).csv"
df_raw = pd.read_csv(URL)
df_raw.head(10)

## Problem Statement

**Problem:**  
Can the three penguin species in the Palmer Archipelago (Adélie, Chinstrap, Gentoo) be reliably distinguished based solely on their morphological measurements (bill length, bill depth, flipper length, and body mass)?

Additionally, several records lack sex assignment. Using group-level morphological comparisons within each species, can we infer the most probable sex for these unclassified individuals?

---

## Research Hypothesis

**Hypothesis:**  
The three penguin species exhibit statistically distinguishable morphological profiles: Gentoo penguins present significantly larger flipper lengths and body mass, while Adélie and Chinstrap differ primarily in bill dimensions. These morphological clusters are strong enough to support species-level classification and sex inference for unclassified records.

---

## Study Justification

Understanding morphological variation across penguin species is critical for:

1. **Ecological monitoring:** Non-invasive species identification from physical measurements supports field research without genetic sampling.
2. **Conservation biology:** Morphological baselines help detect population-level changes driven by climate or habitat shifts in the Antarctic.
3. **Data science education:** The Palmer Penguins dataset serves as a modern, ethically collected alternative to the Iris dataset for teaching classification and EDA techniques.
4. **Missing data imputation:** Developing robust methods to infer missing categorical variables (sex) from continuous measurements has broad methodological applications.

---

### References

- Gorman, K. B., Williams, T. D., & Fraser, W. R. (2014). Ecological sexual dimorphism and environmental variability within a community of Antarctic penguins (genus *Pygoscelis*). *PLoS ONE*, 9(3), e90081. https://doi.org/10.1371/journal.pone.0090081
- Horst, A. M., Hill, A. P., & Gorman, K. B. (2020). *palmerpenguins: Palmer Archipelago (Antarctica) penguin data*. R package version 0.1.0. https://doi.org/10.5281/zenodo.3960218

---

## Initial Data Exploration

---

In [ ]:
print(f"Shape: {df_raw.shape[0]} observations x {df_raw.shape[1]} variables\n")
print("Column types:")
print(df_raw.dtypes)
print(f"\nDuplicate rows: {df_raw.duplicated().sum()}")

In [ ]:
df_raw.replace(".", np.nan, inplace=True)
df_raw.replace("NA", np.nan, inplace=True)

numeric_cols = ["culmen_length_mm", "culmen_depth_mm", "flipper_length_mm", "body_mass_g"]
for col in numeric_cols:
    df_raw[col] = pd.to_numeric(df_raw[col], errors="coerce")

print("Missing values per column:")
print(df_raw.isnull().sum())
print(f"\nTotal missing: {df_raw.isnull().sum().sum()}")
print(f"Percentage of rows with any missing value: {df_raw.isnull().any(axis=1).mean():.1%}")

In [ ]:
missing_sex = df_raw[df_raw["sex"].isnull()]
print(f"Records with missing sex: {len(missing_sex)}")
print(f"\nSpecies breakdown of missing sex:")
print(missing_sex["species"].value_counts())

In [ ]:
df = df_raw.dropna(subset=numeric_cols).copy()
print(f"Working dataset after dropping rows with missing numeric values: {df.shape[0]} rows")
df.describe()

### Initial Observations

- The dataset contains 344 observations across 7 variables (3 categorical, 4 numeric).
- Missing values appear in both numeric measurements and the `sex` column.
- The `sex` column contains records labeled as `NA` that represent our target for inference.
- After cleaning, we retain 342 observations with complete numeric measurements.
- Variable ranges suggest substantial morphological variation across the dataset.

### Variable Selection

| Variable | Type | Description | Justification |
|----------|------|-------------|---------------|
| `species` | Categorical | Penguin species (Adélie, Chinstrap, Gentoo) | Primary grouping variable for morphological comparison |
| `island` | Categorical | Island of origin (Biscoe, Dream, Torgersen) | Captures geographic variation and species distribution |
| `sex` | Categorical | Biological sex (MALE, FEMALE) | Key variable for dimorphism analysis and missing-value inference |
| `culmen_length_mm` | Numeric | Bill length in mm | Discriminates between Adélie/Chinstrap and Gentoo |
| `culmen_depth_mm` | Numeric | Bill depth in mm | Strong species-level separator (Simpson's paradox candidate) |
| `flipper_length_mm` | Numeric | Flipper length in mm | Clear Gentoo separator; correlates with body size |
| `body_mass_g` | Numeric | Body mass in grams | Overall size indicator; strong sexual dimorphism signal |

## Descriptive Statistics

---

In [ ]:
species_stats = df.groupby("species")[numeric_cols].agg(["mean", "median", "std", "min", "max"])
species_stats.columns = [f"{col}_{stat}" for col, stat in species_stats.columns]
species_stats.T

In [ ]:
print("Quartiles by species:\n")
for sp in df["species"].unique():
    print(f"--- {sp} ---")
    print(df[df["species"] == sp][numeric_cols].quantile([0.25, 0.5, 0.75]))
    print()

In [ ]:
print("Species counts:")
print(df["species"].value_counts())
print("\nIsland counts:")
print(df["island"].value_counts())
print("\nSex counts (excluding missing):")
print(df["sex"].value_counts())

### Histograms — Distribution of Numeric Variables by Species

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
titles = ["Culmen Length (mm)", "Culmen Depth (mm)", "Flipper Length (mm)", "Body Mass (g)"]

for ax, col, title in zip(axes.flat, numeric_cols, titles):
    for sp in df["species"].unique():
        subset = df[df["species"] == sp]
        ax.hist(subset[col], bins=20, alpha=0.5, label=sp, edgecolor="black", linewidth=0.5)
    ax.set_title(title, fontsize=13, fontweight="bold")
    ax.set_xlabel(title)
    ax.set_ylabel("Frequency")
    ax.legend()

fig.suptitle("Distribution of Morphological Variables by Species", fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

**Interpretation:** Flipper length and body mass show the clearest species separation — Gentoo forms a distinct cluster at higher values, while Adélie and Chinstrap overlap substantially. Culmen length distributions for Adélie peak lower (~38 mm) with minimal overlap with Chinstrap (~48 mm), suggesting it is a strong discriminator between these two species. Culmen depth reveals an interesting pattern: Gentoo penguins cluster at shallower depths despite their larger overall size.

### Boxplots — Outlier Detection and Group Comparison

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, col, title in zip(axes.flat, numeric_cols, titles):
    sns.boxplot(data=df, x="species", y=col, hue="species", ax=ax, palette="Set2", legend=False)
    ax.set_title(title, fontsize=13, fontweight="bold")
    ax.set_xlabel("Species")
    ax.set_ylabel(title)

fig.suptitle("Morphological Measurements by Species", fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

**Interpretation:** The boxplots confirm that Gentoo penguins have significantly longer flippers (median ~217 mm vs. ~190 mm for Adélie/Chinstrap) and higher body mass (median ~5000 g vs. ~3700 g). A few outliers appear in Adélie body mass (higher than expected) and Chinstrap flipper length. The interquartile ranges for culmen depth are relatively tight within each species, indicating low intra-species variability in this trait.

### Bar Charts — Species Distribution Across Islands

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ct = pd.crosstab(df["island"], df["species"])
ct.plot(kind="bar", ax=axes[0], rot=0, edgecolor="black", linewidth=0.5)
axes[0].set_title("Species Count by Island", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Island")
axes[0].set_ylabel("Count")
axes[0].legend(title="Species")

sex_known = df[df["sex"].isin(["MALE", "FEMALE"])]
ct2 = pd.crosstab(sex_known["species"], sex_known["sex"])
ct2.plot(kind="bar", ax=axes[1], rot=0, color=["#e07b54", "#5b8fa8"], edgecolor="black", linewidth=0.5)
axes[1].set_title("Sex Distribution by Species", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Species")
axes[1].set_ylabel("Count")
axes[1].legend(title="Sex")

plt.tight_layout()
plt.show()

**Interpretation:** Species distribution is geographically structured: Chinstrap penguins are exclusive to Dream Island, Gentoo to Biscoe, while Adélie is the only species present on all three islands. This geographic partitioning introduces potential confounding between island and species effects. The sex distribution within each species is approximately balanced, which supports unbiased sex-based comparisons.

### Scatter Plots — Bivariate Relationships

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

pairs = [
    ("culmen_length_mm", "culmen_depth_mm", "Culmen Length vs. Depth"),
    ("flipper_length_mm", "body_mass_g", "Flipper Length vs. Body Mass"),
    ("culmen_length_mm", "flipper_length_mm", "Culmen Length vs. Flipper Length"),
]

for ax, (x, y, title) in zip(axes, pairs):
    sns.scatterplot(data=df, x=x, y=y, hue="species", style="species", ax=ax, s=60, alpha=0.7)
    ax.set_title(title, fontsize=13, fontweight="bold")
    ax.legend(title="Species", fontsize=9)

plt.tight_layout()
plt.show()

**Interpretation:** The culmen length vs. depth scatter plot reveals Simpson's paradox: the overall trend appears negative, but within each species, the correlation is positive. This is a textbook example of how aggregation can mask group-level relationships. Flipper length and body mass show a strong positive linear relationship, with Gentoo occupying the upper-right cluster. The three species form visually separable clusters across most bivariate projections, supporting the hypothesis that morphology alone can discriminate species.

## Critical Analysis

---

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, col, title in zip(axes.flat, numeric_cols, titles):
    for sp in df["species"].unique():
        subset = df[df["species"] == sp][col].dropna()
        sns.kdeplot(subset, ax=ax, label=sp, fill=True, alpha=0.3)
    ax.set_title(f"Density: {title}", fontsize=13, fontweight="bold")
    ax.set_xlabel(title)
    ax.legend()

fig.suptitle("KDE Distributions by Species", fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

**Interpretation:** The KDE plots confirm that body mass distributions are approximately symmetric for each species, while culmen length for Adélie shows a slight right skew. Gentoo's flipper length distribution is notably narrow (low variability), suggesting strong morphological constraint on this trait. Culmen depth distributions reveal that Gentoo occupies a distinctly lower range (13–16 mm) compared to Adélie and Chinstrap (16–21 mm).

In [ ]:
sex_known = df[df["sex"].isin(["MALE", "FEMALE"])]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, col, title in zip(axes.flat, numeric_cols, titles):
    sns.boxplot(data=sex_known, x="species", y=col, hue="sex", ax=ax, palette="Set2")
    ax.set_title(f"{title} by Species and Sex", fontsize=13, fontweight="bold")
    ax.set_xlabel("Species")
    ax.set_ylabel(title)
    ax.legend(title="Sex", fontsize=9)

fig.suptitle("Sexual Dimorphism in Morphological Traits", fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

**Interpretation:** Sexual dimorphism is consistent across all three species: males are larger than females in every measurement. The dimorphism is most pronounced in body mass (males ~500–800 g heavier) and culmen length (males ~3–5 mm longer). This consistent pattern supports the feasibility of sex inference for unclassified records — individuals with measurements above the species median are more likely male.

In [ ]:
print("Coefficient of Variation (%) by species:\n")
cv = df.groupby("species")[numeric_cols].agg(lambda x: (x.std() / x.mean()) * 100)
cv.columns = [c.replace("_", " ").title() for c in cv.columns]
cv.round(2)

**Interpretation:** Body mass shows the highest coefficient of variation across all species (10–13%), indicating it is the most variable trait relative to its mean. Flipper length is the least variable (~4–6%), meaning it is the most morphologically constrained. Chinstrap shows slightly lower variability than Adélie and Gentoo in bill dimensions, possibly reflecting a more homogeneous population.

### Skewness and Outlier Analysis

In [ ]:
print("Skewness by species:\n")
skew = df.groupby("species")[numeric_cols].skew()
print(skew.round(3))

print("\n\nOutlier count (IQR method) by species:")
for sp in df["species"].unique():
    subset = df[df["species"] == sp]
    outlier_counts = {}
    for col in numeric_cols:
        q1 = subset[col].quantile(0.25)
        q3 = subset[col].quantile(0.75)
        iqr = q3 - q1
        n_outliers = ((subset[col] < q1 - 1.5 * iqr) | (subset[col] > q3 + 1.5 * iqr)).sum()
        outlier_counts[col] = n_outliers
    print(f"\n{sp}: {outlier_counts}")

**Interpretation:** Most distributions exhibit low to moderate skewness (|skew| < 0.5), confirming approximate symmetry. Body mass for Adélie is positively skewed, driven by a few heavier individuals. The IQR method identifies very few outliers per species, suggesting that extreme values are rare and the data is well-behaved. These outliers likely represent natural biological variation rather than measurement errors.

## Relationship Between Variables

---

In [ ]:
g = sns.pairplot(
    df, hue="species", vars=numeric_cols,
    diag_kind="kde", plot_kws={"alpha": 0.6, "s": 40},
    height=2.5
)
g.figure.suptitle("Pairplot: All Numeric Variables by Species", y=1.02, fontsize=15, fontweight="bold")
plt.show()

**Interpretation:** The pairplot confirms that the three species form distinct morphological clusters in most bivariate projections. The strongest species separation occurs in the flipper length–body mass plane, where Gentoo is entirely isolated. Adélie and Chinstrap overlap in flipper length and body mass but separate cleanly along culmen length. The diagonal KDEs reinforce that each variable contributes complementary discriminatory power.

### Correlations

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

pearson_corr = df[numeric_cols].corr(method="pearson")
sns.heatmap(pearson_corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=axes[0],
            square=True, linewidths=0.5)
axes[0].set_title("Pearson Correlation (Overall)", fontsize=13, fontweight="bold")

spearman_corr = df[numeric_cols].corr(method="spearman")
sns.heatmap(spearman_corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=axes[1],
            square=True, linewidths=0.5)
axes[1].set_title("Spearman Correlation (Overall)", fontsize=13, fontweight="bold")

plt.tight_layout()
plt.show()

**Interpretation:** At the overall level, flipper length and body mass are strongly positively correlated (r ≈ 0.87), confirming that larger penguins have proportionally longer flippers. Culmen depth shows a negative overall correlation with flipper length (r ≈ −0.58) and body mass (r ≈ −0.47), but this is driven by species composition (Simpson's paradox). Pearson and Spearman correlations are very similar, indicating approximately linear monotonic relationships.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, sp in zip(axes, df["species"].unique()):
    subset = df[df["species"] == sp][numeric_cols]
    corr = subset.corr(method="pearson")
    sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax,
                square=True, linewidths=0.5, vmin=-1, vmax=1)
    ax.set_title(f"Pearson Correlation — {sp}", fontsize=13, fontweight="bold")

plt.tight_layout()
plt.show()

**Interpretation:** Within-species correlations reveal the true structure: culmen depth is positively correlated with culmen length within each species (reversing the overall negative trend — confirming Simpson's paradox). Gentoo shows the strongest within-species correlation between flipper length and body mass (r ≈ 0.70). Adélie has weaker internal correlations, suggesting more independent variation among its morphological traits. These within-group patterns are essential: correlation does not equal causation, and aggregated correlations can be misleading when groups are not accounted for.

## Sex Inference for Unclassified Records

---

In [ ]:
sex_known = df[df["sex"].isin(["MALE", "FEMALE"])]

species_sex_medians = sex_known.groupby(["species", "sex"])[numeric_cols].median()
print("Median measurements by species and sex:\n")
species_sex_medians

In [ ]:
unclassified = df[~df["sex"].isin(["MALE", "FEMALE"])].copy()
print(f"Unclassified records: {len(unclassified)}\n")

species_sex_means = sex_known.groupby(["species", "sex"])[numeric_cols].mean()

inferred = []
for idx, row in unclassified.iterrows():
    sp = row["species"]
    male_mean = species_sex_means.loc[(sp, "MALE")]
    female_mean = species_sex_means.loc[(sp, "FEMALE")]
    dist_male = np.sqrt(((row[numeric_cols] - male_mean) ** 2).sum())
    dist_female = np.sqrt(((row[numeric_cols] - female_mean) ** 2).sum())
    inferred_sex = "MALE" if dist_male < dist_female else "FEMALE"
    inferred.append({
        "index": idx, "species": sp,
        "culmen_length_mm": row["culmen_length_mm"],
        "body_mass_g": row["body_mass_g"],
        "dist_to_male": round(dist_male, 1),
        "dist_to_female": round(dist_female, 1),
        "inferred_sex": inferred_sex
    })

inferred_df = pd.DataFrame(inferred)
print("Inferred sex for unclassified records:\n")
inferred_df

**Interpretation:** Using Euclidean distance to species-sex group centroids, each unclassified record is assigned to the nearest sex group. This nearest-centroid approach leverages the consistent sexual dimorphism observed across all four measurements. The confidence of each assignment is reflected in the distance differential — larger gaps between male and female distances indicate higher-confidence inferences.

## Limitations

- **Sample size:** With 342 usable records (68 Chinstrap, 146 Adélie, 119 Gentoo after cleaning), subgroup analyses (species × sex × island) have limited statistical power.
- **Missing data:** 11 records lack sex information and 2 lack all numeric measurements. The missing-sex records are not randomly distributed across species, which could introduce bias in sex-inference results.
- **Observational scope:** The dataset captures only four morphological variables. Other traits (e.g., plumage, vocalizations, genetics) are absent but relevant for species discrimination.
- **Temporal confounding:** Data was collected across multiple years; annual variation in body condition is not accounted for.
- **Descriptive limitations:** This EDA identifies associations but cannot establish causal relationships between morphology and species identity. The sex inference method (nearest centroid) is a heuristic; formal classification (e.g., logistic regression, LDA) would provide probabilistic estimates with confidence intervals.
- **Geographic bias:** Species are not uniformly distributed across islands (Chinstrap exclusive to Dream, Gentoo to Biscoe), making it impossible to disentangle species from island effects.

---

## Conclusions

### Does the hypothesis hold?

**Yes.** The three penguin species exhibit clearly distinguishable morphological profiles:

- **Gentoo** penguins are significantly larger in flipper length (median ~217 mm) and body mass (median ~5050 g) compared to Adélie and Chinstrap.
- **Adélie** and **Chinstrap** are similar in body size but diverge in bill dimensions: Chinstrap has longer bills (~48 mm vs. ~38 mm), while Adélie has deeper bills relative to their length.
- These morphological clusters are visually and statistically separable in most bivariate projections.

### Key Patterns

1. **Simpson's paradox** in culmen dimensions: the overall negative correlation between bill length and depth reverses to positive within each species.
2. **Consistent sexual dimorphism**: males are larger across all four measurements in every species, enabling heuristic sex inference.
3. **Geographic partitioning**: species distribution is strongly tied to island identity, a confound that must be addressed in any predictive model.
4. **Low outlier prevalence**: the IQR method detects very few outliers, indicating clean, well-collected data.

### Actionable Insights

- A simple nearest-centroid classifier using all four morphological variables can distinguish species and infer missing sex.
- For formal classification tasks, flipper length and culmen length should be prioritized as features due to their strong discriminatory power.
- Future studies should incorporate temporal (year) and genetic variables to strengthen inference.

---

### Final Reflection

This EDA demonstrates that careful descriptive analysis — when paired with appropriate group-level decomposition — can reveal biological structure that aggregated statistics obscure. The discovery of Simpson's paradox in bill dimensions underscores the danger of naive correlation analysis and highlights the importance of stratified exploration. The consistent sexual dimorphism across species supports practical applications in field ecology, where non-invasive sex determination from morphometrics could reduce the need for genetic sampling.